In [23]:
import keras
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras import layers, models, regularizers
from keras.optimizers import Adam
from keras.optimizers.schedules import CosineDecay
from keras.callbacks import ModelCheckpoint
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
from keras.models import load_model
from keras import Model
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [2]:
def load_physics_dataset(files, features, labels=None, N_particles_max=50):
    events = []
    event_labels = []

    if labels is not None:
        if len(labels) == 1:
            labels = labels * len(files)
        else:
            assert len(labels) == len(files)

    for i, fname in enumerate(files):
        df = pd.read_parquet(fname)
        label = labels[i] if labels is not None else None

        for idx in range(len(df)):
            event_features = []

            for feat in features:
                arr = np.asarray(df[feat].iloc[idx], dtype=np.float32)

                if len(arr) > N_particles_max:
                    arr = arr[:N_particles_max]
                else:
                    padded = np.zeros(N_particles_max, dtype=np.float32)
                    padded[:len(arr)] = arr
                    arr = padded

                event_features.append(arr)

            event_tensor = np.stack(event_features, axis=1)
            events.append(event_tensor)

            if label is not None:
                event_labels.append(label)

    X = np.asarray(events, dtype=np.float32)
    y = np.asarray(event_labels, dtype=np.int64) if labels is not None else None

    return X, y



def preprocess_particles(X, feature_indices=[0,1,2,3], pt_index=0, eta_index=1, phi_index=2,
                         d0_index=3, eta_range=(-5.0, 5.0)):

    X_proc = X.copy()

    pt = X_proc[:, :, pt_index]
    pt_logged = np.log(pt + 1.0)
    pt_min, pt_max = pt_logged.min(), pt_logged.max()
    X_proc[:, :, pt_index] = (pt_logged - pt_min) / (pt_max - pt_min + 1e-8)

    eta = X_proc[:, :, eta_index]
    eta_min, eta_max = eta_range
    X_proc[:, :, eta_index] = (eta - eta_min) / (eta_max - eta_min)
    X_proc[:, :, eta_index] = np.clip(X_proc[:, :, eta_index], 0.0, 1.0)

    d0 = X_proc[:, :, d0_index]
    d0_min, d0_max = d0.min(), d0.max()
    X_proc[:, :, d0_index] = (d0 - d0_min) / (d0_max - d0_min + 1e-8)

    phi = X_proc[:, :, phi_index]
    phi_cos = np.cos(phi)
    phi_sin = np.sin(phi)

    X_proc = np.delete(X_proc, phi_index, axis=2)
    X_proc = np.concatenate([X_proc, phi_cos[..., np.newaxis], phi_sin[..., np.newaxis]], axis=2)

    return X_proc


def load_test_data(df, features, labels=None, N_particles_max=50):

    events = []

    if labels is not None:
        assert len(labels) == len(df), "Labels must match number of events"

    for idx, row in df.iterrows():
        event_features = []
        for feat in features:
            arr = np.asarray(row[feat], dtype=np.float32)
            if len(arr) > N_particles_max:
                arr = arr[:N_particles_max]
            else:
                padded = np.zeros(N_particles_max, dtype=np.float32)
                padded[:len(arr)] = arr
                arr = padded
            event_features.append(arr)
        event_tensor = np.stack(event_features, axis=1)  # shape (N_particles, N_features)
        events.append(event_tensor)

    X = np.asarray(events, dtype=np.float32)
    y = np.asarray(labels, dtype=np.int64) if labels is not None else None
    return X, y

In [3]:
CERNBOX_TOKEN = "cK4cbOHaCwfYUKy"
CERNBOX_BASE  = f"https://cernbox.cern.ch/remote.php/dav/public-files/{CERNBOX_TOKEN}"

In [4]:
files = [
    CERNBOX_BASE+'/HH_4b.parquet',
    CERNBOX_BASE+'/DYJetsToLL_13TeV-madgraphMLM-pythia8.parquet',
    CERNBOX_BASE+'/QCD_HT50toInf.parquet',
    CERNBOX_BASE+'/WJetsToLNu_13TeV-madgraphMLM-pythia8.parquet'
]

features = [            #The features per particle in the event, Kinematics + displacement from PV
    'L1T_PFPart_PT',
    'L1T_PFPart_Eta',
    'L1T_PFPart_Phi',
    'L1T_PFPart_D0'
]


labels = [1,0,0,0]   # HH=1, DY=2, QCD=0, WJets=3

X, y = load_physics_dataset(
    files,
    features,
    labels=labels,
    N_particles_max=16
)

print(X.shape)  # (N total events, N particles per event, N features per particle)
print(y.shape)  # (N total events)

(230985, 16, 4)
(230985,)


In [5]:
X_proc = preprocess_particles(X) #Here, we are normalising the input data to put features on the same scale. Because Phi is periodic, we encode as sine and cosine so we expand the feature dimension
print(X_proc.shape)

(230985, 16, 5)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.4, random_state=42, stratify=y, shuffle = True
)

X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.2, random_state=42, stratify=y_test, shuffle = True
)

In [24]:
class TransformerBlock(layers.Layer):

    def __init__(self,embed_dim,num_heads,ff_dim,dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=None):

        attention_output = self.attention(x,x,training=training)

        x = self.norm1(
            x + self.dropout1(attention_output,training=training)
        )

        ffn_output = self.ffn(x,training=training)

        x = self.norm2(
            x + self.dropout2(ffn_output,training=training)
        )

        return x

In [26]:
n_particles = X_train.shape[1] # 16
n_features = X_train.shape[2] # 5

inputs = keras.Input(
    shape=(n_particles, n_features)
)

x = layers.Dense(256)(inputs)

for _ in range(6):

    x = TransformerBlock(
        embed_dim=256,
        num_heads=8,
        ff_dim=1024
    )(x)

x = layers.GlobalAveragePooling1D()(x)

x = layers.Dense(128, activation="gelu")(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(64, activation="gelu")(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(16, activation="gelu")(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(1, activation="sigmoid")(x)
x = layers.Dropout(0.1)(x)

model = models.Model(inputs, x)

model.compile(
    # optimizer=optimizer, 
    loss='binary_crossentropy', 
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC')
    ]
)

checkpoint_path = "mlp_best_model.h5"
checkpoint_cb = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64
)
model.evaluate(
    X_test,
    y_test,
    batch_size=64
)

Epoch 1/20


2026-09-03 23:47:43.919813: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907
2026-09-03 23:47:54.391102: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_161', 1956 bytes spill stores, 1964 bytes spill loads

2026-09-03 23:47:56.982386: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_161', 712 bytes spill stores, 500 bytes spill loads

2026-09-03 23:47:57.064386: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_161', 184 bytes spill stores, 184 bytes spill loads

2026-09-03 23:47:57.287676: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_f

  25/2166 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.6764 - auc: 0.4964 - loss: 1.1195   

I0000 00:00:1788479297.786379   20665 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2163/2166 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7381 - auc: 0.5014 - loss: 0.9448

...

2026-09-03 23:49:10.548384: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_45', 4 bytes spill stores, 4 bytes spill loads

2026-09-03 23:49:11.488471: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_45', 36 bytes spill stores, 36 bytes spill loads

2026-09-03 23:49:11.639740: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1262', 4 bytes spill stores, 4 bytes spill loads

2026-09-03 23:49:12.154981: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_45', 16 bytes spill stores, 16 bytes spill loads

2026-09-03 23:49:12.538601: I external/local_xla/xla/stream_executor/c

2166/2166 ━━━━━━━━━━━━━━━━━━━━ 102s 27ms/step - accuracy: 0.7398 - auc: 0.5021 - loss: 0.9307 - val_accuracy: 0.7404 - val_auc: 0.5000 - val_loss: 0.5726
Epoch 2/20
2166/2166 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.7404 - auc: 0.5000 - loss: 0.9423 - val_accuracy: 0.7404 - val_auc: 0.5000 - val_loss: 0.5751
Epoch 3/20
2166/2166 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.7401 - auc: 0.5024 - loss: 0.9361 - val_accuracy: 0.7404 - val_auc: 0.5000 - val_loss: 0.5761
Epoch 4/20
2166/2166 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.7401 - auc: 0.5021 - loss: 0.9258 - val_accuracy: 0.7404 - val_auc: 0.5000 - val_loss: 0.5745
Epoch 5/20
2166/2166 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.7404 - auc: 0.4986 - loss: 0.9397 - val_accuracy: 0.7404 - val_auc: 0.5000 - val_loss: 0.5730
Epoch 6/20
2166/2166 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.7404 - auc: 0.5009 - loss: 0.9396 - val_accuracy: 0.7404 - val_auc: 0.5000 - val_loss: 0.5728
Epoch 7/20
2166/2166 ━━━━━━━━━━

2026-09-03 23:54:17.883159: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_45', 16 bytes spill stores, 16 bytes spill loads

2026-09-03 23:54:18.859862: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_45', 36 bytes spill stores, 36 bytes spill loads

2026-09-03 23:54:19.201618: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_45', 4 bytes spill stores, 4 bytes spill loads



1155/1155 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.7404 - auc: 0.5000 - loss: 0.5749


[0.5749005079269409, 0.740431547164917, 0.5]

In [ ]:
def evaluate_model(model, X, y, model_name="Model"):
    y_pred = np.argmax(model.predict(X, batch_size=512), axis=1)

    acc = np.mean(y_pred == y)
    print(f"\n=== {model_name} ===")
    print(f"Test Accuracy: {acc:.3f}\n")

    print("Classification Report:")
    print(classification_report(y, y_pred, digits=3))

    cm = confusion_matrix(y, y_pred)
    print("Confusion Matrix:")
    print(cm)

    return acc, y_pred, cm

In [1]:
mlp = load_model("mlp_best_model.h5")
#transformer = load_model("transformer_best_model.keras", custom_objects={'CLSToken': CLSToken, 'ExtractCLS': ExtractCLS})

NameError: name 'load_model' is not defined

In [ ]:
mlp_acc, mlp_pred, mlp_cm = evaluate_model(mlp, X_test, y_test, "MLP")

In [ ]:
mlp_embedding = Model(
    inputs=mlp.input,
    outputs=mlp.layers[-2].output
)
mlp_embedding  = mlp_embedding.predict(X_test)

In [ ]:
def plot_embeddings(embeddings, labels, title):
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)
    plt.figure(figsize=(6,5))
    for cls in np.unique(labels):
        mask = labels == cls
        plt.scatter(emb_2d[mask,0], emb_2d[mask,1], label=f"Class {cls}", s=10, alpha=0.1)
    plt.title(title)
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_embeddings(mlp_embedding, y_test, "MLP Internal Representations")